# Tarea 4: Heart Failure Prediction: Validación, Hiperparámetros y Comparación de Modelos - Parte 2. 

MDS7104: Aprendizaje de Maquinas - Otoño 2026

---

### Cuerpo Docente:

- Profesor: Francisco Vásquez L.
- Auxiliares: Álvaro Márquez y Diego Olguín Wende
- Ayudantes: Javiera Yañez y Tamara Carrasco


### Estudiante

- Felipe Muñoz M.

In [ ]:
!uv add numpy requests pandas matplotlib scipy scikit-learn ruff pre-commit

Resolved 63 packages in 20ms
Audited 58 packages in 16ms


In [3]:
!uv add ipykernel ipython matplotlib-inline

Resolved 130 packages in 24ms
Audited 127 packages in 26ms


In [1]:
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

In [2]:
import plotly.graph_objects as go

In [37]:
# %matplotlib inline

# b)Código Base Tarea 3

Consideraciones:
* Clase positiva `HeartDiseare=1`
* Se generan variables dummies$^2$ con `drop_first=True` para evitar colinealidad perfecta
* Se fija la misma semilla (`seed=7`) y la misma partición 80/20 estratificada empleada en la Tarea 3, así los resultados de ambas tareas serán directamente comparables
* Se considera el siguiente conjunto de métricas: `M ={Accuracy, Precision, Recall, Specificity, F1-Score, FPR, FNR}`

In [3]:
# Semilla global
SEED = 7

# Carga de datos
df = pd.read_csv("/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/tareas/tarea3/heart.csv")

In [4]:
# Preprocesamiento: dummies con drop_first=True
df_dummies = pd.get_dummies(df, drop_first=True)

X = df_dummies.drop(columns="HeartDisease").values
y = df_dummies["HeartDisease"].values
feature_names = df_dummies.drop(columns="HeartDisease").columns.tolist()

# Partición 80/20 estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

# Modelos baseline
# Regresión Logística (mejor configuración hallada en Tarea 3)
lr_best = LogisticRegression(
    C=10,
    class_weight={0: 1, 1: 5},
    solver="saga",
    l1_ratio=1.0,
    max_iter=2000,
    random_state=SEED,
)
lr_best.fit(X_train, y_train)

# LDA (covarianza compartida, todas las variables)
lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train)


# Función de métricas M
def metricas_M(y_true, y_pred, nombre="") -> dict:
    """
    Calcula el conjunto M = {Accuracy, Precision, Recall, Specificity,
    F1-Score, FPR, FNR} e imprime un resumen.

    Retorna un dict con las métricas para uso programático.
    """
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    metricas = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),  # TPR
        "Specificity": tn / (tn + fp),  # TNR
        "F1-Score": f1_score(y_true, y_pred, zero_division=0),
        "FPR": fp / (fp + tn),
        "FNR": fn / (fn + tp),
    }
    if nombre:
        print(f"\n── Métricas {nombre} ──────────────────────────")
        for k, v in metricas.items():
            print(f"  {k:<12}: {v:.4f}")
    return metricas

/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


# P1 - Teoría de la información y selección de modelos

## C) Selección de variables vía información mutua

In [5]:
# ── P1(c): Información mutua Î(Xj; Y) ────────────────────────────────────────
from sklearn.feature_selection import mutual_info_classif

In [6]:
# Estimación empírica de IM sobre el conjunto de entrenamiento
mi_scores = mutual_info_classif(
    X_train,
    y_train,
    discrete_features=False,
    random_state=SEED,
)

# Ranking de variables por IM
df_mi = pd.DataFrame({"variable": feature_names, "IM": mi_scores}).sort_values("IM", ascending=False).reset_index(drop=True)
df_mi.index += 1  # ranking desde 1
print("── Ranking de variables por Î(Xj; Y) ──────────────────────────────")
print(df_mi.to_string())

── Ranking de variables por Î(Xj; Y) ──────────────────────────────
             variable        IM
1         ST_Slope_Up  0.221238
2       ST_Slope_Flat  0.170494
3    ExerciseAngina_Y  0.127002
4             Oldpeak  0.110984
5               MaxHR  0.087934
6         Cholesterol  0.078007
7   ChestPainType_ATA  0.072163
8               Sex_M  0.063442
9   ChestPainType_NAP  0.036584
10                Age  0.033682
11      RestingECG_ST  0.025048
12          FastingBS  0.017485
13          RestingBP  0.012824
14   ChestPainType_TA  0.012560
15  RestingECG_Normal  0.004345


Se definen los tres subconjuntos
- `M1`: Son todas las variables del baseline completo
- `M2`: top-k variables con mayor IM
    - Las primeras 5 variables concentran la mayor parte de la IM total
    - `k=5` mantiene parsimonia frente a M1 (que tiene ~15 columnas tras dummiesy sigue siendo comparable con el número de features "informativas" que suelen surgir en datasets cardíacos de esta dimensión.
    - A partir del 6° lugar la IM cae notoriamente (codo en el ranking), lo que sugiere que las variables adicionales aportan poco.
- `M3`: Subconjunto contrastante — variables exclusivamente numéricas originales
    - Las variables numéricas (Age, RestingBP, Cholesterol, FastingBS, MaxHR,Oldpeak) forman un subconjunto semánticamente coherente y permiten comparar el poder predictivo de la información continua pura contra la información de las dummies categóricas.
    - Al mismo tiempo, algunas de estas variables suelen tener IM baja (e.g. RestingBP, Cholesterol), por lo que M3 actúa como contraste "débil" respecto a M2.
    - Alternativa posible: las k variables con menor IM, pero ese conjunto resulta difícil de interpretar clínicamente.

In [7]:
# Definición de los tres subconjuntos

# M1: todas las variables (baseline completo)
vars_M1 = feature_names

# M2: top-k variables con mayor IM
K = 5
vars_M2 = df_mi["variable"].iloc[:K].tolist()

# M3: subconjunto contrastante — variables exclusivamente numéricas originales
vars_M3 = [f for f in feature_names if f in ["Age", "RestingBP", "Cholesterol", "FastingBS", "MaxHR", "Oldpeak"]]

print(f"\nM1 ({len(vars_M1)} variables): todas")
print(f"M2 ({len(vars_M2)} variables): {vars_M2}")
print(f"M3 ({len(vars_M3)} variables): {vars_M3}")


M1 (15 variables): todas
M2 (5 variables): ['ST_Slope_Up', 'ST_Slope_Flat', 'ExerciseAngina_Y', 'Oldpeak', 'MaxHR']
M3 (6 variables): ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']


In [8]:
# Índices de columna para cada subconjunto
idx_M1 = [feature_names.index(v) for v in vars_M1]
idx_M2 = [feature_names.index(v) for v in vars_M2]
idx_M3 = [feature_names.index(v) for v in vars_M3]

X_train_M1, X_test_M1 = X_train[:, idx_M1], X_test[:, idx_M1]
X_train_M2, X_test_M2 = X_train[:, idx_M2], X_test[:, idx_M2]
X_train_M3, X_test_M3 = X_train[:, idx_M3], X_test[:, idx_M3]

# Gráfico de barras del ranking
colores = ["#08306b"] * K + ["#6baed6"] * (len(vars_M2) - K if K > len(vars_M2) else 0)
colores = (
    ["#2ca02c"] * K  # top-k (M2) → verde
    + ["#aec7e8"] * (len(feature_names) - K)  # resto → azul claro
)

fig = go.Figure(
    go.Bar(
        x=df_mi["IM"],
        y=df_mi["variable"],
        orientation="h",
        marker_color=colores,
        text=df_mi["IM"].round(4),
        textposition="outside",
    )
)

fig.update_layout(
    title=dict(text="Î(Xj; Y) — Información mutua con HeartDisease (train)", x=0.5, xanchor="center"),
    xaxis_title="Información mutua estimada",
    yaxis=dict(autorange="reversed"),
    height=520,
    width=720,
    margin=dict(l=160, r=80, t=60, b=50),
)
fig.show()

## d) Cálculo de AIC y BIC para Regresión Lógistica. Para cada $M\in \{M_1,M_2,M_3\}$

Justificación de $C=10^6$: `sklearn.LogisticRegression` minimiza  $\frac{1}{C} \cdot \|\beta\|^{2} - \ell (\beta)$. Con $C=10^6$, la penalización L2 es muy cercana a 0, por lo tanto es equivalente a estimar los coeficientes mediante EMV sin regularización. Esto hace que $\hat{\ell}$ sea comparable entre modelos con distinto número de variables, lo cual es condición necesaria para que AIC y BIC tengan sentido como criterio de selección de modelos. 

In [9]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [ ]:
C_MV = 1 * (10**6)  # valor grande para aproximar MV (sin regularización efectiva)
CV_K = 5  # número de folds para Cross Validation estratificada
skf = StratifiedKFold(n_splits=CV_K, shuffle=True, random_state=SEED)

# Subconjuntos definidos en P1(c)
subsets = {
    "M1": (X_train_M1, X_test_M1, vars_M1),
    "M2": (X_train_M2, X_test_M2, vars_M2),
    "M3": (X_train_M3, X_test_M3, vars_M3),
}

N_train = X_train.shape[0]  # tamaño del conjunto de entrenamiento

resultados_lr = {}

for nombre, (X_tr, X_te, vars_) in subsets.items():  # noqa: B007
    # (i) Ajuste por MV (C grande, sin regularización efectiva) ────────────────
    modelo = LogisticRegression(
        C=C_MV,
        solver="lbfgs",
        max_iter=5000,
        random_state=SEED,
    )
    modelo.fit(X_tr, y_train)

    # (ii) Log-verosimilitud ˆl sobre entrenamiento
    # predict_log_proba devuelve log P(Y=0|x) y log P(Y=1|x) por fila.
    # Sumamos la log-prob de la clase observada para cada observación.
    log_proba = modelo.predict_log_proba(X_tr)  # (N_train, 2)
    l_hat = log_proba[np.arange(N_train), y_train].sum()

    # (iii) Número de parámetros d (coeficientes + intercepto)
    d = modelo.coef_.shape[1] + 1  # M features + 1 intercepto

    # (iv) AIC y BIC
    # AIC = 2d − 2ˆl
    # BIC = d·ln(N) − 2ˆl
    aic = 2 * d - 2 * l_hat
    bic = d * np.log(N_train) - 2 * l_hat

    # (v) F1-Score por CV estratificada k=5 sobre entrenamiento
    # Usamos el mismo C_MV para no mezclar efectos de regularización.
    modelo_cv = LogisticRegression(
        C=C_MV,
        solver="lbfgs",
        max_iter=5000,
        random_state=SEED,
    )
    f1_cv_scores = cross_val_score(
        modelo_cv,
        X_tr,
        y_train,
        cv=skf,
        scoring="f1",
    )
    f1_cv = f1_cv_scores.mean()

    resultados_lr[nombre] = {
        "l_hat": l_hat,
        "d": d,
        "AIC": aic,
        "BIC": bic,
        "F1-CV": f1_cv,
        "modelo": modelo,  # guardamos para P1(f)
    }

    print(f"\n── {nombre} ({len(vars_)} variables) ───────────────────────────")
    print(f"  l          : {l_hat:.4f}")
    print(f"  d           : {d}")
    print(f"  AIC         : {aic:.4f}")
    print(f"  BIC         : {bic:.4f}")
    print(f"  F1-CV (k=5) : {f1_cv:.4f}  ±{f1_cv_scores.std():.4f}")


── M1 (15 variables) ───────────────────────────
  l          : -235.1858
  d           : 16
  AIC         : 502.3715
  BIC         : 575.9477
  F1-CV (k=5) : 0.8801  ±0.0228

── M2 (5 variables) ───────────────────────────
  l          : -303.9941
  d           : 6
  AIC         : 619.9881
  BIC         : 647.5792
  F1-CV (k=5) : 0.8464  ±0.0185

── M3 (6 variables) ───────────────────────────
  l          : -355.5894
  d           : 7
  AIC         : 725.1788
  BIC         : 757.3683
  F1-CV (k=5) : 0.7875  ±0.0366


In [11]:
# ── Tabla resumen (se completa en P1(f) junto con LDA)
df_lr = pd.DataFrame(
    {
        m: {
            "l": v["l_hat"],
            "d": v["d"],
            "AIC": v["AIC"],
            "BIC": v["BIC"],
            "F1-CV": v["F1-CV"],
        }
        for m, v in resultados_lr.items()
    }
).T.round(4)

print("\n── Tabla resumen Regresión Logística ───────────────────────────────")
print(df_lr.to_string())


── Tabla resumen Regresión Logística ───────────────────────────────
           l     d       AIC       BIC   F1-CV
M1 -235.1858  16.0  502.3715  575.9477  0.8801
M2 -303.9941   6.0  619.9881  647.5792  0.8464
M3 -355.5894   7.0  725.1788  757.3683  0.7875


## e) Cálculo de AIC y BIC para LDA. Para M1, con $k=5$

Se procede a calcular la Log-verosmilitud conjunta $\hat{\ell}$ siguiendo la siguiente expresión obtenida:
- $\hat{\ell} =  N1·ln(π̂) + N0·ln(1-π̂) - (N/2)·ln|Σ̂| - (N·M/2)·(1 + ln(2π))$


In [13]:
# Log-verosimilitud conjunta ˆl
N_train = X_train.shape[0]
M = X_train.shape[1]  # número de variables explicativas (M=15)
K = 2  # clasificación binaria

# Priors estimados por LDA
pi_hat = lda.priors_  # [π̂_0, π̂_1]
N0 = int(round(pi_hat[0] * N_train))
N1 = int(round(pi_hat[1] * N_train))

# Determinar la covarianza compartida estimada
# lda.covariance_ contiene Σ̂. Se usa sign y logdet para evitar underflow numérico
lda = LinearDiscriminantAnalysis(store_covariance=True)  # Se ajusta la matriz de covartianza compartida
lda.fit(X_train, y_train)

_, log_det_sigma = np.linalg.slogdet(lda.covariance_)

# Término de los priors
term_priors = N1 * np.log(pi_hat[1]) + N0 * np.log(pi_hat[0])

# Término de la covarianza
term_cov = -(N_train / 2) * log_det_sigma

# Término constante gaussiano
term_const = -(N_train * M / 2) * (1 + np.log(2 * np.pi))

l_hat_lda = term_priors + term_cov + term_const

print(f"Términos de ˆl:")  # noqa: F541
print(f"  Priors     : {term_priors:.4f}")
print(f"  Covarianza : {term_cov:.4f}")
print(f"  Constante  : {term_const:.4f}")
print(f"  ˆl total   : {l_hat_lda:.4f}")

Términos de ˆl:
  Priors     : -504.6178
  Covarianza : -1276.6154
  Constante  : -15622.5133
  ˆl total   : -17403.7464


Se procede a calculaar el número de parametros d, dada la siguiente formula encontrada
- $d = (K-1) + K·M + M(M+1)/2$

In [14]:
# Número de parámetros d
d_lda = (K - 1) + K * M + M * (M + 1) // 2
print("\nd = (K-1) + K·M + M(M+1)/2")
print(f"  = {K - 1} + {K * M} + {M * (M + 1) // 2} = {d_lda}")


d = (K-1) + K·M + M(M+1)/2
  = 1 + 30 + 120 = 151


Se calcula el AIC, BIC y F1 del modelo LDA

In [15]:
# AIC y BIC
aic_lda = 2 * d_lda - 2 * l_hat_lda
bic_lda = d_lda * np.log(N_train) - 2 * l_hat_lda

print(f"\nAIC (LDA) : {aic_lda:.4f}")
print(f"BIC (LDA) : {bic_lda:.4f}")

# F1-CV estratificada k=5
lda_cv = LinearDiscriminantAnalysis()
f1_cv_scores_lda = cross_val_score(
    lda_cv,
    X_train,
    y_train,
    cv=skf,
    scoring="f1",
)
f1_cv_lda = f1_cv_scores_lda.mean()

print(f"F1-CV (k=5): {f1_cv_lda:.4f}  ±{f1_cv_scores_lda.std():.4f}")


AIC (LDA) : 35109.4928
BIC (LDA) : 35803.8677
F1-CV (k=5): 0.8802  ±0.0228


## f) Tabla comparativa

In [16]:
# Agregar al diccionario de resultados para P1(f)
resultados_lr["LDA"] = {
    "l_hat": l_hat_lda,
    "d": d_lda,
    "AIC": aic_lda,
    "BIC": bic_lda,
    "F1-CV": f1_cv_lda,
    "modelo": lda,
}

# Tabla comparativa completa (M1, M2, M3, LDA)
df_todos = pd.DataFrame(
    {
        m: {
            "ˆl": v["l_hat"],
            "d": v["d"],
            "AIC": v["AIC"],
            "BIC": v["BIC"],
            "F1-CV": v["F1-CV"],
        }
        for m, v in resultados_lr.items()
    }
).T.round(4)

print("\n── Tabla comparativa completa ──────────────────────────────────────")
print(df_todos.to_string())


── Tabla comparativa completa ──────────────────────────────────────
             ˆl      d         AIC         BIC   F1-CV
M1    -235.1858   16.0    502.3715    575.9477  0.8801
M2    -303.9941    6.0    619.9881    647.5792  0.8464
M3    -355.5894    7.0    725.1788    757.3683  0.7875
LDA -17403.7464  151.0  35109.4928  35803.8677  0.8802


In [17]:
X_train.shape[0]

734

Se procede a calcular las diferencias de AIC y BIC

In [18]:
#  ΔAIC y ΔBIC

modelos = ["M1", "M2", "M3", "LDA"]

aic_vals = {m: resultados_lr[m]["AIC"] for m in modelos}
bic_vals = {m: resultados_lr[m]["BIC"] for m in modelos}

aic_min = min(aic_vals.values())
bic_min = min(bic_vals.values())

delta_aic = {m: aic_vals[m] - aic_min for m in modelos}
delta_bic = {m: bic_vals[m] - bic_min for m in modelos}

# Umbrales Burnham & Anderson (2004) para ΔAIC
# ΔAIC ∈ [0, 2)   → apoyo sustancial
# ΔAIC ∈ [2, 7)   → apoyo considerablemente menor
# ΔAIC ∈ [7, 10)  → apoyo débil
# ΔAIC ≥ 10       → esencialmente sin apoyo


def nivel_aic(delta):
    if delta < 2:
        return "Apoyo sustancial"
    elif delta < 7:
        return "Apoyo considerablemente menor"
    elif delta < 10:
        return "Apoyo débil"
    else:
        return "Sin apoyo"


# Umbrales Kass & Raftery (1995) para ΔBIC
# ΔBIC ∈ [0, 2)   → evidencia no significativa contra M*
# ΔBIC ∈ [2, 6)   → evidencia positiva contra M_i
# ΔBIC ∈ [6, 10)  → evidencia fuerte contra M_i
# ΔBIC ≥ 10       → evidencia muy fuerte contra M_i


def nivel_bic(delta):
    if delta < 2:
        return "No significativa"
    elif delta < 6:
        return "Positiva"
    elif delta < 10:
        return "Fuerte"
    else:
        return "Muy fuerte"


# Tabla de resultados
print(f"{'Modelo':<6} {'AIC':>10} {'ΔAIC':>10} {'Nivel ΔAIC':<35} {'BIC':>10} {'ΔBIC':>10} {'Nivel ΔBIC'}")
print("-" * 100)
for m in modelos:
    print(f"{m:<6} {aic_vals[m]:>10.3f} {delta_aic[m]:>10.3f} {nivel_aic(delta_aic[m]):<35} {bic_vals[m]:>10.3f} {delta_bic[m]:>10.3f} {nivel_bic(delta_bic[m])}")

Modelo        AIC       ΔAIC Nivel ΔAIC                                 BIC       ΔBIC Nivel ΔBIC
----------------------------------------------------------------------------------------------------
M1        502.372      0.000 Apoyo sustancial                       575.948      0.000 No significativa
M2        619.988    117.617 Sin apoyo                              647.579     71.632 Muy fuerte
M3        725.179    222.807 Sin apoyo                              757.368    181.421 Muy fuerte
LDA     35109.493  34607.121 Sin apoyo                            35803.868  35227.920 Muy fuerte


# P2 - Support Vector Machines

## b) SVM con hiperparámetros por defecto

In [19]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

Notar que el kernel RBF calcula $k(x, x') = \exp(-\gamma\|x - x'\|^2)$, por lo que es sensible a la escala, esto implica que, variables con rangos grandes dominarán la distancia euclidiana y distorsionarán el kernel. Por tanto, es necesario estandarizar las variables numéricas; las variables dummy $(0/1)$  no requieren escala adicional. Para evitar data leakage, se ajusta usa `StandardScaler` exclusivamente sobre el conjunto de train y se aplica la transformación a ambos splits.

In [20]:
# Identificar columnas numéricas y dummy
# Las dummies ya están en 0/1; solo escalar las continuas
cols_dummy = [i for i, name in enumerate(feature_names) if df_dummies[name].nunique() == 2]
cols_num = [i for i in range(X.shape[1]) if i not in cols_dummy]

# Preprocesamiento sin data leakage
scaler = StandardScaler()
X_train_sc = X_train.copy().astype(float)
X_test_sc = X_test.copy().astype(float)

# Fit SOLO en train, transform en ambos
X_train_sc[:, cols_num] = scaler.fit_transform(X_train[:, cols_num])
X_test_sc[:, cols_num] = scaler.transform(X_test[:, cols_num])

# SVC con hiperparámetros por defecto
svc_default = SVC(random_state=SEED)  # kernel='rbf', C=1.0, gamma='scale'
svc_default.fit(X_train_sc, y_train)
y_pred_svc = svc_default.predict(X_test_sc)

# Métricas
metricas_svc = metricas_M(y_test, y_pred_svc, nombre="SVC (default, RBF)")


── Métricas SVC (default, RBF) ──────────────────────────
  Accuracy    : 0.8804
  Precision   : 0.8571
  Recall      : 0.9412
  Specificity : 0.8049
  F1-Score    : 0.8972
  FPR         : 0.1951
  FNR         : 0.0588


In [22]:
# Predicciones en train y test
y_pred_svc_train = svc_default.predict(X_train_sc)
y_pred_svc_test = svc_default.predict(X_test_sc)

# Métricas M en ambos splits
metricas_svc_train = metricas_M(y_train, y_pred_svc_train, nombre="SVC Train")
metricas_svc_test = metricas_M(y_test, y_pred_svc_test, nombre="SVC Test")

# Tabla comparativa
df_metricas_svc = pd.DataFrame(
    {
        "Train": metricas_svc_train,
        "Test": metricas_svc_test,
    }
).round(4)
print("\n── Comparativa Train vs Test ────────────────────────────────────")
print(df_metricas_svc.to_string())


── Métricas SVC Train ──────────────────────────
  Accuracy    : 0.9005
  Precision   : 0.8899
  Recall      : 0.9360
  Specificity : 0.8567
  F1-Score    : 0.9124
  FPR         : 0.1433
  FNR         : 0.0640

── Métricas SVC Test ──────────────────────────
  Accuracy    : 0.8804
  Precision   : 0.8571
  Recall      : 0.9412
  Specificity : 0.8049
  F1-Score    : 0.8972
  FPR         : 0.1951
  FNR         : 0.0588

── Comparativa Train vs Test ────────────────────────────────────
              Train    Test
Accuracy     0.9005  0.8804
Precision    0.8899  0.8571
Recall       0.9360  0.9412
Specificity  0.8567  0.8049
F1-Score     0.9124  0.8972
FPR          0.1433  0.1951
FNR          0.0640  0.0588


In [30]:
from plotly.subplots import make_subplots


def plot_cm_doble(y_true_train, y_pred_train, y_true_test, y_pred_test):
    labels = ["No Enf.(0)", "Enf. (1)"]
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=["Matriz de Confusión – Train", "Matriz de Confusión – Test"],
        horizontal_spacing=0.15,
    )

    for col, (y_true, y_pred, color) in enumerate(
        [
            (y_true_train, y_pred_train, "Blues"),
            (y_true_test, y_pred_test, "Reds"),
        ],
        start=1,
    ):
        cm = confusion_matrix(y_true, y_pred)
        fig.add_trace(
            go.Heatmap(
                z=cm[::-1],
                x=labels,
                y=labels[::-1],
                colorscale=color,
                text=cm[::-1],
                texttemplate="%{text}",
                textfont=dict(size=16),
                showscale=False,
            ),
            row=1,
            col=col,
        )

    fig.update_layout(
        width=860,
        height=400,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
    )
    fig.update_xaxes(title_text="Predicho")
    fig.update_yaxes(title_text="Real", col=1)

    fig.show()
    fig.write_image("confusion_matrix_svc_default.png", scale=2)


plot_cm_doble(y_train, y_pred_svc_train, y_test, y_pred_svc_test)

Se construye la curva ROC-AUC

In [31]:
from sklearn.metrics import auc, roc_curve

# Scores via decision_function (no probabilidades)
scores_svc = svc_default.decision_function(X_test_sc)

# Curva ROC y AUC
fpr_svc, tpr_svc, _ = roc_curve(y_test, scores_svc)
auc_svc = auc(fpr_svc, tpr_svc)

# AUC Regresión Logística (Tarea 3) — reemplaza con tu valor real
auc_lr = 0.9191  # <-- ajusta con el valor que reportaste en T3

# Curva ROC con Plotly
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=fpr_svc,
        y=tpr_svc,
        mode="lines",
        name=f"SVC RBF (AUC = {auc_svc:.3f})",
        line=dict(color="crimson", width=2),
    )
)

fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        name="Clasificador aleatorio",
        line=dict(color="gray", width=1.5, dash="dash"),
    )
)

fig.update_layout(
    title="Curva ROC — SVC (default, RBF) vs. Regresión Logística (T3)",
    xaxis_title="Tasa de Falsos Positivos (FPR)",
    yaxis_title="Tasa de Verdaderos Positivos (TPR)",
    legend=dict(x=0.6, y=0.1),
    width=600,
    height=450,
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
)
fig.update_xaxes(showgrid=True, gridcolor="lightgrey")
fig.update_yaxes(showgrid=True, gridcolor="lightgrey")

fig.show()
fig.write_image("roc_svc_default.png", scale=2)

print(f"AUC SVC  (RBF)         : {auc_svc:.4f}")
print(f"AUC Reg. Logística (T3): {auc_lr:.4f}")

AUC SVC  (RBF)         : 0.9408
AUC Reg. Logística (T3): 0.9191


## c) Optimización de hiperparámetros con GridSearchCV

In [45]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Grid de hiperparámetros
param_grid = [
    # Kernels sin gamma relevante
    {
        "kernel": ["linear"],
        "C": [0.1, 1, 10],
        "class_weight": [None, "balanced", {0: 1, 1: 5}],
    },
    # Kernels con gamma
    {
        "kernel": ["rbf", "poly", "sigmoid"],
        "C": [0.1, 1, 10],
        "class_weight": [None, "balanced", {0: 1, 1: 5}],
        "gamma": ["scale", "auto"],
    },
]

# GridSearchCV con CV estratificado K=5
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

grid_search = GridSearchCV(
    estimator=SVC(random_state=SEED),
    param_grid=param_grid,
    scoring="f1",  # criterio principal: F1-Score
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True,  # reajusta con mejores params sobre todo X_train_sc
)

grid_search.fit(X_train_sc, y_train)

# (i) Mejores hiperparámetros
print("\n── Mejores hiperparámetros ──────────────────────────────────────")
print(grid_search.best_params_)
print(f"F1-Score CV (train)   : {grid_search.best_score_:.4f}")

Fitting 5 folds for each of 63 candidates, totalling 315 fits

── Mejores hiperparámetros ──────────────────────────────────────
{'C': 1, 'class_weight': None, 'gamma': 'scale', 'kernel': 'rbf'}
F1-Score CV (train)   : 0.8849


Se selecciona el mejor modelo y se reportan sus métricas

In [46]:
# Mejor modelo
svc_best = grid_search.best_estimator_

In [47]:
# Métricas en train y test

y_pred_best_train = svc_best.predict(X_train_sc)
y_pred_best_test = svc_best.predict(X_test_sc)

scores_best_train = svc_best.decision_function(X_train_sc)
scores_best_test = svc_best.decision_function(X_test_sc)

metricas_best_train = metricas_M(y_train, y_pred_best_train, nombre="SVC Óptimo Train")
metricas_best_test = metricas_M(y_test, y_pred_best_test, nombre="SVC Óptimo Test")

auc_best_train = roc_auc_score(y_train, scores_best_train)
auc_best_test = roc_auc_score(y_test, scores_best_test)

print(f"\nAUC Train (óptimo): {auc_best_train:.4f}")
print(f"AUC Test  (óptimo): {auc_best_test:.4f}")


── Métricas SVC Óptimo Train ──────────────────────────
  Accuracy    : 0.9005
  Precision   : 0.8899
  Recall      : 0.9360
  Specificity : 0.8567
  F1-Score    : 0.9124
  FPR         : 0.1433
  FNR         : 0.0640

── Métricas SVC Óptimo Test ──────────────────────────
  Accuracy    : 0.8804
  Precision   : 0.8571
  Recall      : 0.9412
  Specificity : 0.8049
  F1-Score    : 0.8972
  FPR         : 0.1951
  FNR         : 0.0588

AUC Train (óptimo): 0.9587
AUC Test  (óptimo): 0.9408


In [48]:
# Tabla comparativa default vs óptimo
# AUC default (ya calculado en parte b)
auc_default_train = roc_auc_score(y_train, svc_default.decision_function(X_train_sc))

comparativa = pd.DataFrame(
    {
        "SVC Default Train": {**metricas_svc_train, "AUC": auc_default_train},
        "SVC Default Test": {**metricas_svc_test, "AUC": auc_svc},
        "SVC Óptimo Train": {**metricas_best_train, "AUC": auc_best_train},
        "SVC Óptimo Test": {**metricas_best_test, "AUC": auc_best_test},
    }
).T.round(4)

# Mostrar solo métricas relevantes
print("\n── Comparativa Default vs Óptimo ────────────────────────────────")
print(comparativa[["Accuracy", "F1-Score", "Recall", "Specificity", "AUC"]].to_string())


── Comparativa Default vs Óptimo ────────────────────────────────
                   Accuracy  F1-Score  Recall  Specificity     AUC
SVC Default Train    0.9005    0.9124  0.9360       0.8567  0.9587
SVC Default Test     0.8804    0.8972  0.9412       0.8049  0.9408
SVC Óptimo Train     0.9005    0.9124  0.9360       0.8567  0.9587
SVC Óptimo Test      0.8804    0.8972  0.9412       0.8049  0.9408


## e) Comparación integradora

Se incorporan al dataset de métricas, las metricas calculadas para SVM

In [ ]:
# Mapeo de X_test por modelo
X_test_map = {
    "M1": X_test_M1,
    "M2": X_test_M2,
    "M3": X_test_M3,
    "LDA": X_test,  # LDA usa todas las variables
}

# Construir tabla final
filas = {}

for nombre, v in resultados_lr.items():
    modelo = v["modelo"]
    X_te = X_test_map[nombre]

    y_pred = modelo.predict(X_te)
    y_score = modelo.predict_proba(X_te)[:, 1]

    filas[nombre] = {
        "F1-CV": v["F1-CV"],
        "F1-test": f1_score(y_test, y_pred, zero_division=0),
        "AUC-test": roc_auc_score(y_test, y_score),
        "AIC": v["AIC"],
        "BIC": v["BIC"],
    }

# Agregar SVM best
filas[r"SVM_best"] = {
    "F1-CV": grid_search.best_score_,
    "F1-test": metricas_best_test["F1-Score"],
    "AUC-test": roc_auc_score(y_test, svc_best.decision_function(X_test_sc)),
    "AIC": None,
    "BIC": None,
}

# DataFrame y display
df_final = pd.DataFrame(filas).T

df_final_display = df_final.copy().round(4)
df_final_display["AIC"] = df_final_display["AIC"].apply(lambda x: f"{x:.2f}" if pd.notna(x) else "N/A")
df_final_display["BIC"] = df_final_display["BIC"].apply(lambda x: f"{x:.2f}" if pd.notna(x) else "N/A")

print("\n── Tabla comparativa final ──────────────────────────────────────────")
print(df_final_display[["F1-CV", "F1-test", "AUC-test", "AIC", "BIC"]].to_string())


── Tabla comparativa final ──────────────────────────────────────────
           F1-CV  F1-test  AUC-test       AIC       BIC
M1        0.8801   0.8544    0.9260    502.37    575.95
M2        0.8464   0.8230    0.8885    619.99    647.58
M3        0.7875   0.8173    0.8460    725.18    757.37
LDA       0.8802   0.8654    0.9287  35109.49  35803.87
SVM_best  0.8849   0.8972    0.9408       N/A       N/A
